In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
import math

In [16]:
df = pd.read_csv('Social_Network_Ads.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   UserID           400 non-null    int64 
 1   Gender           400 non-null    object
 2   Age              400 non-null    int64 
 3   EstimatedSalary  400 non-null    int64 
 4   Purchased        400 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 15.8+ KB


In [17]:
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

df = df.drop(columns=['UserID'])
print(df)

     Gender  Age  EstimatedSalary  Purchased
0         1   19            19000          0
1         1   35            20000          0
2         0   26            43000          0
3         0   27            57000          0
4         1   19            76000          0
..      ...  ...              ...        ...
395       0   46            41000          1
396       1   51            23000          1
397       0   50            20000          1
398       1   36            33000          0
399       0   49            36000          1

[400 rows x 4 columns]


In [22]:
train = df.sample(frac=0.8,random_state=222)
test = df.drop(train.index)

X_train = train.drop(columns=['Purchased'])
Y_train = train['Purchased']
X_test = test.drop(columns=['Purchased'])
Y_test = test['Purchased']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
def calculate_stats(X, y):
    classes = list(set(y))
    stats = {}
    for c in classes:
        X_c = [X[i] for i in range(len(y)) if y[i] == c]
        prior = len(X_c) / len(y)
        feature_stats = []
        for col in range(len(X[0])):
            vals = [row[col] for row in X_c]
            mean = sum(vals)/len(vals)
            var = sum((x - mean) ** 2 for x in vals)/len(vals) + 1e-9
            feature_stats.append((mean, var))
        stats[c] = {"prior": prior,"features": feature_stats}
    return stats
def predict_single(sample,stats):
    best_class = None
    max_log_posterior = -float('inf')
    for c, c_info in stats.items():
        log_posterior = math.log(c_info["prior"])
        for col,x in enumerate(sample):
            mean,var = c_info["features"][col]
            log_pdf = -0.5 * math.log(2*math.pi*var)-((x - mean) ** 2 / (2 * var))
            log_posterior += log_pdf
        if log_posterior > max_log_posterior:
            max_log_posterior = log_posterior
            best_class = c
    return best_class

X_train_list = X_train_scaled.tolist()
y_train_list = Y_train.tolist()

X_test_list = X_test_scaled.tolist()
y_test_list = Y_test.tolist()

model_stats = calculate_stats(X_train_list, y_train_list)
y_pred = [predict_single(sample, model_stats) for sample in X_test_list]

tp = sum(1 for i in range(len(y_test_list)) if y_test_list[i] == 1 and y_pred[i] == 1)
tn = sum(1 for i in range(len(y_test_list)) if y_test_list[i] == 0 and y_pred[i] == 0)
fp = sum(1 for i in range(len(y_test_list)) if y_test_list[i] == 0 and y_pred[i] == 1)
fn = sum(1 for i in range(len(y_test_list)) if y_test_list[i] == 1 and y_pred[i] == 0)

accuracy = (tp + tn) / len(y_test_list)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"Accuracy:{accuracy}")
print(f"Precision:{precision}")
print(f"Recall:{recall}")
print(f"F1-Score:{f1_score}")
print(f"True Negatives (TN): {tn:<6}|False Negatives (FN): {fn}")
print(f"False Positives (FP): {fp:<5}|True Positives (TP): {tp}")



Accuracy:0.8625
Precision:0.875
Recall:0.7241379310344828
F1-Score:0.7924528301886793
True Negatives (TN): 48    |False Negatives (FN): 8
False Positives (FP): 3    |True Positives (TP): 21
